<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/TCGA_COAD_BRAF_Mutations_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 TCGA-COAD × BRAF Mutation Extractor (v4)
Extract all **BRAF** mutations from the **TCGA Colon Adenocarcinoma (COAD)** study
using the [cBioPortal](https://www.cbioportal.org) public REST API.

**v4 fix:** Switched from `POST /mutations/fetch` (unreliable — returns empty body) to the stable
`GET /molecular-profiles/{profileId}/mutations?sampleListId=...&projection=DETAILED` endpoint.


In [ ]:
!pip install requests pandas matplotlib -q

In [ ]:
import requests, json, csv
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_URL    = "https://www.cbioportal.org/api"
STUDY_ID    = "coadread_tcga_pub"
GENE_SYMBOL = "BRAF"
OUTPUT_CSV  = "tcga_coad_braf_mutations.csv"

print(f"Study : {STUDY_ID}  |  Gene : {GENE_SYMBOL}")


In [ ]:
# ── API helpers ────────────────────────────────────────────────────────────────

def _check(resp, label):
    """Raise a clear RuntimeError if the response is bad."""
    if not resp.ok:
        raise RuntimeError(
            f"[{label}] HTTP {resp.status_code}\n"
            f"URL : {resp.url}\n"
            f"Body: {resp.text[:600] or '(empty)'}"
        )
    if not resp.text.strip():
        raise RuntimeError(
            f"[{label}] Empty body (HTTP {resp.status_code})\n"
            f"URL: {resp.url}"
        )
    try:
        return resp.json()
    except Exception:
        raise RuntimeError(
            f"[{label}] Non-JSON body\nURL: {resp.url}\nBody: {resp.text[:600]}"
        )


def get_mutation_profile_id(study_id):
    resp = requests.get(
        f"{BASE_URL}/studies/{study_id}/molecular-profiles",
        headers={"Accept": "application/json"}, timeout=30
    )
    data = _check(resp, "molecular-profiles")
    for p in data:
        if p.get("molecularAlterationType") == "MUTATION_EXTENDED":
            return p["molecularProfileId"]
    raise RuntimeError(f"No MUTATION_EXTENDED profile in {study_id}")


def get_entrez_gene_id(symbol):
    resp = requests.get(
        f"{BASE_URL}/genes/{symbol}",
        headers={"Accept": "application/json"}, timeout=15
    )
    data = _check(resp, "genes")
    return data["entrezGeneId"]


def get_sample_list_id(study_id):
    """Return the canonical all-sample list id for a study."""
    resp = requests.get(
        f"{BASE_URL}/studies/{study_id}/sample-lists",
        headers={"Accept": "application/json"}, timeout=30
    )
    data = _check(resp, "sample-lists")
    for sl in data:
        if sl.get("sampleListId") == f"{study_id}_all":
            return sl["sampleListId"]
    # fallback: biggest list
    best = max(data, key=lambda x: x.get("sampleCount", 0))
    print(f"  ⚠️  Using fallback sample list: {best['sampleListId']}")
    return best["sampleListId"]


def fetch_mutations_get(profile_id, sample_list_id, entrez_gene_id):
    """
    Use the stable GET endpoint:
      GET /molecular-profiles/{profileId}/mutations
          ?sampleListId=...&entrezGeneId=...&projection=DETAILED

    This is the same call the cBioPortal frontend itself makes.
    Falls back to paginated fetch if needed.
    """
    params = {
        "sampleListId":  sample_list_id,
        "entrezGeneId":  entrez_gene_id,
        "projection":    "DETAILED",
        "pageSize":      10000,
        "pageNumber":    0,
        "sortBy":        "proteinChange",
        "direction":     "ASC",
    }
    resp = requests.get(
        f"{BASE_URL}/molecular-profiles/{profile_id}/mutations",
        params=params,
        headers={"Accept": "application/json"},
        timeout=60,
    )
    return _check(resp, "mutations GET")


print("✅  API helpers ready")


In [ ]:
# ── Flatten ────────────────────────────────────────────────────────────────────

def flatten_mutation(m):
    gene = m.get("gene", {})
    return {
        "sampleId":              m.get("sampleId", ""),
        "patientId":             m.get("patientId", ""),
        "studyId":               m.get("studyId", ""),
        "molecularProfileId":    m.get("molecularProfileId", ""),
        "uniqueSampleKey":       m.get("uniqueSampleKey", ""),
        "uniquePatientKey":      m.get("uniquePatientKey", ""),
        "hugoGeneSymbol":        gene.get("hugoGeneSymbol", ""),
        "entrezGeneId":          gene.get("entrezGeneId", ""),
        "mutationType":          m.get("mutationType", ""),
        "proteinChange":         m.get("proteinChange", ""),
        "aminoAcidChange":       m.get("aminoAcidChange", ""),
        "chromosome":            m.get("chr", ""),
        "startPosition":         m.get("startPosition", ""),
        "endPosition":           m.get("endPosition", ""),
        "referenceAllele":       m.get("referenceAllele", ""),
        "variantAllele":         m.get("variantAllele", ""),
        "ncbiBuild":             m.get("ncbiBuild", ""),
        "variantType":           m.get("variantType", ""),
        "functionalImpactScore": m.get("functionalImpactScore", ""),
        "validationStatus":      m.get("validationStatus", ""),
        "mutationStatus":        m.get("mutationStatus", ""),
        "sequencingCenter":      m.get("center", ""),
        "tumorRefCount":         m.get("tumorRefCount", ""),
        "tumorAltCount":         m.get("tumorAltCount", ""),
        "normalRefCount":        m.get("normalRefCount", ""),
        "normalAltCount":        m.get("normalAltCount", ""),
        "keyword":               m.get("keyword", ""),
        "refseqMrnaId":          m.get("refseqMrnaId", ""),
        "proteinPosStart":       m.get("proteinPosStart", ""),
        "proteinPosEnd":         m.get("proteinPosEnd", ""),
    }

print("✅  flatten_mutation ready")


In [ ]:
# ── Run extraction ─────────────────────────────────────────────────────────────

print(f"🔍  Fetching {GENE_SYMBOL} mutations from {STUDY_ID} …\n")

print("[1/4] Resolving mutation profile …")
profile_id = get_mutation_profile_id(STUDY_ID)
print(f"      ✅  {profile_id}")

print(f"[2/4] Resolving Entrez Gene ID for {GENE_SYMBOL} …")
entrez_id = get_entrez_gene_id(GENE_SYMBOL)
print(f"      ✅  Entrez ID = {entrez_id}")

print("[3/4] Fetching all-sample list …")
sample_list_id = get_sample_list_id(STUDY_ID)
print(f"      ✅  sampleListId = {sample_list_id}")

print(f"[4/4] Fetching {GENE_SYMBOL} mutations via GET endpoint …")
raw_mutations = fetch_mutations_get(profile_id, sample_list_id, entrez_id)
print(f"      ✅  Raw records returned: {len(raw_mutations)}")

rows = [flatten_mutation(m) for m in raw_mutations]
df   = pd.DataFrame(rows)
print(f"\n🎉  DataFrame shape: {df.shape}  ({df['patientId'].nunique()} unique patients)")


In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────

print(f"{'='*65}")
print(f"  TCGA-COAD | {GENE_SYMBOL} Mutations | Total records : {len(df)}")
print(f"{'='*65}")
print(f"  Unique patients : {df['patientId'].nunique()}")
print(f"  Unique samples  : {df['sampleId'].nunique()}")

print("\n  Top protein changes:")
for change, cnt in df["proteinChange"].value_counts().head(10).items():
    print(f"    {change:<22}  {cnt:>4}")

print("\n  Mutation types:")
for mt, cnt in df["mutationType"].value_counts().items():
    print(f"    {mt:<32}  {cnt:>4}")
print(f"{'='*65}")


In [ ]:
# ── Interactive table preview ──────────────────────────────────────────────────

cols = ["sampleId", "patientId", "proteinChange", "mutationType",
        "chromosome", "startPosition", "referenceAllele", "variantAllele",
        "mutationStatus", "validationStatus"]
display(df[cols].head(20))


In [ ]:
# ── Save CSV + download ────────────────────────────────────────────────────────

df.to_csv(OUTPUT_CSV, index=False)
print(f"✅  Saved {len(df)} rows → {OUTPUT_CSV}")

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print("⬇️   Download triggered.")
except ImportError:
    print(f"(Not in Colab — file saved locally as '{OUTPUT_CSV}')")


In [ ]:
# ── Bar chart: top protein changes ────────────────────────────────────────────

top = df["proteinChange"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(top.index, top.values, color="#e74c3c", edgecolor="white")
ax.set_title(f"Top {GENE_SYMBOL} Protein Changes — TCGA-COAD  (n={len(df)} mutations)", fontsize=14)
ax.set_xlabel("Protein Change")
ax.set_ylabel("Count")
ax.bar_label(bars, padding=3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("braf_protein_changes.png", dpi=150)
plt.show()
